# Lab 5
- **Student Name**: Daniel Alejandro Castillo Martin
- **Student ID**: N10019284
- **Course:** Advanced Deep Learning - AIGC-5500-0NA
- **Professor**: Hossein Pourmodheji
---

This lab implements scaled dot-product and multi-head attention from scratch and verifies them against PyTorch. Experiments compare attention with an LSTM on delayed recall, evaluate positional encodings on an order task, and examine normalization and computational cost. Results across three seeds are used to assess each approach and its limitations.

## 1. Imports and Setup

### 1.1 Libraries

In [15]:
# Import required libraries
import math
import platform
import sys
from pathlib import Path
from time import perf_counter

import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

# Lab starter module
from scripts.lab05_starter import train_recall, train_order

### 1.2 Constants

In [16]:
# Save experiment tables and figures.
OUTPUT_DIR = Path('output')
OUTPUT_DIR.mkdir(exist_ok=True)

# Use the three seeds required by the lab.
SEEDS = (0, 1, 2)

### 1.3 Reproducibility and Options

In [17]:
# Set the seed for reproducibility
torch.manual_seed(0)

# Set the display options for pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.precision', 4)

In [18]:
# Validate the environment configuration
print(f'Python: {platform.python_version()}')
print(f'PyTorch: {torch.__version__}')
print(f'Interpreter: {sys.executable}')
print(f'Working directory: {Path.cwd()}')
print(f'CPU threads: {torch.get_num_threads()}')
print(f'Seeds: {SEEDS}')

Python: 3.14.6
PyTorch: 2.14.1+cpu
Interpreter: c:\Users\dancm\Documents\humber-aigc-postgraduate\aigc-5500-advanced-deep-learning\labs\lab-05\env\Scripts\python.exe
Working directory: c:\Users\dancm\Documents\humber-aigc-postgraduate\aigc-5500-advanced-deep-learning\labs\lab-05
CPU threads: 12
Seeds: (0, 1, 2)


## 2. Attention from Scratch and Verification

### 2.1 Scaled Dot-Product Attention

Attention compares queries with keys, scales the scores, and applies softmax over the key positions. The resulting weights produce a weighted sum of values.

A boolean mask uses True for allowed positions and False for blocked positions.

In [20]:
def scaled_dot_product_attention(q, k, v, mask=None):
    # Compare each query with every key and scale the scores.
    scores = (q @ k.transpose(-2, -1)) / math.sqrt(q.shape[-1])

    if mask is not None:
        assert mask.dtype == torch.bool, 'The mask must be boolean.'
        assert mask.any(dim = -1).all(), 'Each query needs an allowed key.'

        # Block positions BEFORE applying softmax.
        scores = scores.masked_fill(~mask, float('-inf'))

    # Normalize over keys: each query gets a distribution over key positions.
    weights = torch.softmax(scores, dim = -1)

    # Verify the probability and masking requirements.
    row_sums = weights.sum(dim = -1)
    assert torch.allclose(
        row_sums,
        torch.ones_like(row_sums),
        atol = 0.000001,
        rtol = 0.00001,
    ), 'Attention weight rows must sum to 1.'

    if mask is not None:
        blocked = (~mask).expand_as(weights)
        assert torch.all(
            weights.masked_select(blocked) == 0
        ), 'Masked positions must have exactly zero weight.'

    output = weights @ v
    return output, weights

### 2.2 Verification Against PyTorch

The custom attention function is compared with PyTorch using identical random inputs, with and without a causal mask. The causal mask allows each query to attend only to its own position and earlier positions.

In [25]:
attention_checks = []

for seed in SEEDS:
    generator = torch.Generator().manual_seed(seed)

    # Shape: (batch, sequence length, feature dimension).
    q = torch.randn(2, 5, 8, generator=generator)
    k = torch.randn(2, 5, 8, generator=generator)
    v = torch.randn(2, 5, 6, generator=generator)

    # True means attention is allowed.
    causal_mask = torch.ones(5, 5, dtype=torch.bool).tril()

    for name, mask in [('Unmasked', None), ('Causal', causal_mask),]:
        custom_output, weights = scaled_dot_product_attention(q, k, v, mask)

        reference_output = F.scaled_dot_product_attention(q, k, v, attn_mask = mask, dropout_p = 0.0)

        torch.testing.assert_close(custom_output, reference_output, atol = 0.000001, rtol=0.00001)

        attention_checks.append({
            'seed': seed,
            'case': name,
            'max_absolute_error': (custom_output - reference_output).abs().max().item(),
            'passed': True
            })

attention_checks_df = pd.DataFrame(attention_checks)

attention_checks_df.to_csv(OUTPUT_DIR / 'attention_function_checks.csv', index = False)

print('All attention function checks passed.')

All attention function checks passed.


In [26]:
# Show the attention checks dataframe
attention_checks_df

,seed,case,max_absolute_error,passed
0,0,Unmasked,1.7881e-07,True
1,0,Causal,1.1921e-07,True
2,1,Unmasked,1.7881e-07,True
3,1,Causal,3.5763e-07,True
4,2,Unmasked,1.1921e-07,True
5,2,Causal,1.1921e-07,True


### 2.3 Multi-Head Attention

Multi-head attention projects inputs into queries, keys, and values, splits their features across heads, and applies attention independently within each head. The head outputs are concatenated and passed through an output projection.

An optional context supplies keys and values for cross-attention.

In [23]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()

        assert n_heads > 0, 'n_heads must be positive.'
        assert d_model % n_heads == 0, 'd_model must be divisible by n_heads.'

        self.d_model = d_model
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads

        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)

    def split_heads(self, x):
        batch_size, length, _ = x.shape

        # (B, T, D) -> (B, T, H, Dh) -> (B, H, T, Dh)
        return x.reshape(batch_size, length, self.n_heads, self.head_dim).transpose(1, 2)

    def forward(self, x, context = None, mask = None):
        # Self-attention uses x for queries, keys, and values.
        context = x if context is None else context

        q = self.split_heads(self.q_proj(x))
        k = self.split_heads(self.k_proj(context))
        v = self.split_heads(self.v_proj(context))

        head_outputs, _ = scaled_dot_product_attention(q, k, v, mask)

        batch_size, _, query_length, _ = head_outputs.shape

        # (B, H, Tq, Dh) -> (B, Tq, H, Dh) -> (B, Tq, D)
        combined = head_outputs.transpose(1, 2).reshape(batch_size, query_length, self.d_model)

        return self.out_proj(combined)

### 2.4 Multi-Head Attention Verification

The custom module is compared with PyTorch's MultiheadAttention after copying the same projection weights and biases. Checks cover self-attention, causal self-attention, and cross-attention with different query and context lengths.

In [28]:
mha_checks = []

for seed in SEEDS:
    torch.manual_seed(seed)

    custom_mha = MultiHeadAttention(d_model = 32, n_heads = 4)
    reference_mha = nn.MultiheadAttention(embed_dim = 32, num_heads = 4, dropout = 0.0, batch_first = True)

    # PyTorch stores the Q, K, and V projections in one stacked tensor.
    with torch.no_grad():
        reference_mha.in_proj_weight.copy_(torch.cat([custom_mha.q_proj.weight, custom_mha.k_proj.weight, custom_mha.v_proj.weight,], dim = 0))
        reference_mha.in_proj_bias.copy_(torch.cat([custom_mha.q_proj.bias, custom_mha.k_proj.bias, custom_mha.v_proj.bias, ], dim = 0))
        reference_mha.out_proj.weight.copy_(custom_mha.out_proj.weight)
        reference_mha.out_proj.bias.copy_(custom_mha.out_proj.bias)

    custom_mha.eval()
    reference_mha.eval()

    generator = torch.Generator().manual_seed(seed + 1000)
    x = torch.randn(2, 5, 32, generator = generator)
    context = torch.randn(2, 7, 32, generator = generator)
    causal_mask = torch.ones(5, 5, dtype = torch.bool).tril()

    cases = [('Self-attention', None, None), ('Causal self-attention', None, causal_mask), ('Cross-attention', context, None)]

    with torch.no_grad():
        for name, ctx, mask in cases:
            custom_output = custom_mha(x, context = ctx, mask = mask)

            kv = x if ctx is None else ctx

            # nn.MultiheadAttention uses True for BLOCKED positions.
            blocked_mask = None if mask is None else ~mask

            reference_output, _ = reference_mha(x, kv, kv, attn_mask = blocked_mask, need_weights = False)

            torch.testing.assert_close(custom_output, reference_output, atol = 0.000001, rtol = 0.00001)

            mha_checks.append({
                'seed': seed,
                'case': name,
                'query_length': x.shape[1],
                'key_length': kv.shape[1],
                'max_absolute_error': (custom_output - reference_output).abs().max().item(),
                'passed': True
                })

mha_checks_df = pd.DataFrame(mha_checks)

mha_checks_df.to_csv(OUTPUT_DIR / 'multihead_attention_checks.csv', index = False,)

print('All multi-head attention checks passed.')

All multi-head attention checks passed.


In [29]:
# Show the multi-head attention checks dataframe
mha_checks_df

,seed,case,query_length,key_length,max_absolute_error,passed
0,0,Self-attention,5,5,5.9605e-08,True
1,0,Causal self-attention,5,5,8.9407e-08,True
2,0,Cross-attention,5,7,5.9605e-08,True
3,1,Self-attention,5,5,5.9605e-08,True
4,1,Causal self-attention,5,5,7.4506e-08,True
5,1,Cross-attention,5,7,1.1921e-07,True
6,2,Self-attention,5,5,7.6368e-08,True
7,2,Causal self-attention,5,5,7.4506e-08,True
8,2,Cross-attention,5,7,5.9605e-08,True


### 2.5 Parameter Counts

Parameter counts are measured at a fixed model width of 32 using 1, 2, 4, and 8 heads.

In [30]:
parameter_counts = []

for n_heads in (1, 2, 4, 8):
    model = MultiHeadAttention(d_model = 32, n_heads = n_heads)

    parameter_counts.append({
        'd_model': 32,
        'n_heads': n_heads,
        'head_dim': model.head_dim,
        'parameters': sum(p.numel() for p in model.parameters())
        })

parameter_counts_df = pd.DataFrame(parameter_counts)

parameter_counts_df.to_csv(OUTPUT_DIR / 'attention_parameter_counts.csv', index = False)

In [31]:
# Show the parameter counts dataframe
parameter_counts_df

,d_model,n_heads,head_dim,parameters
0,32,1,32,4224
1,32,2,16,4224
2,32,4,8,4224
3,32,8,4,4224


At a fixed model width, increasing the number of heads changes how features are partitioned but leaves the four projection layers unchanged, so the parameter count remains 4,224.

### 2.6 Permutation Equivariance

Without positional encodings or a mask, shuffling input positions should shuffle the output positions in the same way. This property is checked using the same layer and parameters for both inputs.

In [32]:
permutation_checks = []

for seed in SEEDS:
    torch.manual_seed(seed)

    model = MultiHeadAttention(d_model = 32, n_heads = 4)
    model.eval()

    generator = torch.Generator().manual_seed(seed + 2000)
    x = torch.randn(2, 7, 32, generator = generator)
    permutation = torch.randperm(7, generator = generator)

    with torch.no_grad():
        original_output = model(x)
        shuffled_output = model(x[:, permutation, :])

        expected_output = original_output[:, permutation, :]

        torch.testing.assert_close(shuffled_output, expected_output, atol = 0.000001, rtol = 0.00001)

    permutation_checks.append({
        'seed': seed,
        'max_absolute_error': (shuffled_output - expected_output).abs().max().item(),
        'passed': True
        })

permutation_checks_df = pd.DataFrame(permutation_checks)

permutation_checks_df.to_csv(OUTPUT_DIR / 'attention_permutation_checks.csv', index = False)

print('All permutation equivariance checks passed.')

All permutation equivariance checks passed.


In [33]:
# Show the permutation checks dataframe
permutation_checks_df

,seed,max_absolute_error,passed
0,0,1.1921e-07,True
1,1,8.9407e-08,True
2,2,1.1921e-07,True
